In [1]:
!pip install -q transformers==4.44.0 pyarrow==17.0.0 pydantic==2.11.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 92.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 49.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.6/442.6 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
sigstore-models 0.0.5 requires pydantic>=2.11.7, but you have pydanti

In [1]:
import pandas as pd
import os

# --- Load dataset ---
file_path = "/kaggle/input/cleaned-blog-authorship-corpus/dataset.csv"
df = pd.read_csv(file_path)
print(f"✅ Loaded dataset with {len(df)} rows")

# --- Create output directory for splits ---
output_dir = "/kaggle/working/splits"
os.makedirs(output_dir, exist_ok=True)

# --- Split into 10 roughly equal parts ---
num_parts = 10
chunk_size = len(df) // num_parts

for i in range(num_parts):
    start = i * chunk_size
    end = (i + 1) * chunk_size if i < num_parts - 1 else len(df)
    df_part = df.iloc[start:end]
    output_path = f"{output_dir}/dataset_part_{i+1}.csv"
    df_part.to_csv(output_path, index=False)
    print(f"✅ Saved part {i+1} with {len(df_part)} rows to {output_path}")

print("\nAll splits saved successfully.")


✅ Loaded dataset with 96199 rows
✅ Saved part 1 with 9619 rows to /kaggle/working/splits/dataset_part_1.csv
✅ Saved part 2 with 9619 rows to /kaggle/working/splits/dataset_part_2.csv
✅ Saved part 3 with 9619 rows to /kaggle/working/splits/dataset_part_3.csv
✅ Saved part 4 with 9619 rows to /kaggle/working/splits/dataset_part_4.csv
✅ Saved part 5 with 9619 rows to /kaggle/working/splits/dataset_part_5.csv
✅ Saved part 6 with 9619 rows to /kaggle/working/splits/dataset_part_6.csv
✅ Saved part 7 with 9619 rows to /kaggle/working/splits/dataset_part_7.csv
✅ Saved part 8 with 9619 rows to /kaggle/working/splits/dataset_part_8.csv
✅ Saved part 9 with 9619 rows to /kaggle/working/splits/dataset_part_9.csv
✅ Saved part 10 with 9628 rows to /kaggle/working/splits/dataset_part_10.csv

All splits saved successfully.


In [3]:
n=6
df = pd.read_csv(f"/kaggle/working/splits/dataset_part_{n}.csv")
df.head()

,id,gender,age,industry,text,word_count,age_group
0,3581210,male,33,Finance & Property,i had an interesting conversation with my dad ...,662,Group3 (30–48)
1,3581210,male,33,Finance & Property,"if anything, korea is a country of extremes. e...",387,Group3 (30–48)
2,3581210,male,33,Finance & Property,take a read of this news article from <url> jo...,386,Group3 (30–48)
3,3581210,male,33,Finance & Property,"ah, the korean language...<ELONG>it looks so d...",296,Group3 (30–48)
4,3581210,male,33,Finance & Property,if you click on my profile you'll make a not-s...,499,Group3 (30–48)


In [3]:
# --- Imports ---
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Model setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()

# --- Batched Summarization ---
@torch.no_grad()
@torch.no_grad()
def batch_summarize_texts(texts, max_input_len=1024, max_output_len=256, batch_size=32):
    """Summarize a list of non-empty strings. Returns a list of equal length."""
    summaries = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]  # assume already filtered to valid strings
        inputs = tokenizer(
            batch,
            max_length=max_input_len,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        try:
            summary_ids = model.generate(
                **inputs,
                num_beams=5,
                length_penalty=1.0,       # reduce penalty → allows longer output
                max_length=256,            # increase upper bound
                min_length=150,            # require at least ~200 tokens (≈200 words)
                no_repeat_ngram_size=3,
                early_stopping=False,      # let it reach full length
                repetition_penalty=1.1
            )



            
            decoded = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print("⚠️ OOM detected, retrying with smaller batch size (8)...")
                if device == "cuda":
                    torch.cuda.empty_cache()
                return batch_summarize_texts(texts, max_input_len, max_output_len, batch_size=8)
            else:
                raise

        summaries.extend(decoded)

        # progress for filtered set
        if (i + len(batch)) % 10 == 0 or (i + len(batch)) == total:
            print(f"Processed {i + len(batch)}/{total} summarized rows")

    return summaries


def add_summary_column(df, text_column="text", batch_size=16):
    texts = df[text_column].tolist()

    # mask: summarize only long, non-empty strings
    def is_long(t):
        return isinstance(t, str) and t.strip() and (len(t.split()) > 200)

    mask = [is_long(t) for t in texts]

    # extract only the texts we will summarize
    to_summarize = [t for t, m in zip(texts, mask) if m]

    # run summarization only on the filtered subset
    summarized_subset = batch_summarize_texts(to_summarize, batch_size=batch_size)

    # stitch back to full length
    summaries_full = []
    it = iter(summarized_subset)
    for m, original in zip(mask, texts):
        if m:
            s = next(it)
            summaries_full.append(s if s.strip() else original)
        else:
            # keep original text for short/empty rows
            summaries_full.append(original)

    # lengths now match df length exactly
    df = df.copy()
    df["summary"] = summaries_full
    return df


# --- Run summarization ---
df = add_summary_column(df, batch_size=32)
output_path = f"/kaggle/working/dataset_with_xsum_summary_{n}.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Saved summarized dataset to {output_path}")

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Processed 160/198 summarized rows
Processed 198/198 summarized rows

✅ Saved summarized dataset to /kaggle/working/dataset_with_xsum_summary.csv
